# Process to Download United Health Files for Indiana (or other State) each Month.
# Very Rough Draft Code, use with caution.
### Richard Meraz (rfmeraz@iu.edu)

In [ ]:
#Import Libraries

import itertools
import json
import pathlib
from multiprocessing import Pool
from urllib.parse import parse_qs, urlparse
import os
import duckdb
import pandas 
import subprocess
import pandas
import io
import tabula
from matplotlib import pyplot
import pandas as pd
import glob
from tqdm import tqdm

In [ ]:
os.getcwd()

In [ ]:
!pip install tabula

In [ ]:
#Set Working Directory

os.chdir("/N/u/singrama/Carbonate/Documents/TIC/Richard_Exp")
os.getcwd()

In [ ]:
#Function to Convert Column Names to Lowercase

def qdf(con, sql):
    """Helper function to execute sql on pandas.DataFrames with duckdb"""
    df = con.execute(sql).df()
    df.columns = df.columns.str.lower()
    return df

con = duckdb.connect()

In [ ]:
# Read in Company Reference Information to map to Employer's EINs

def get_refdata():

    # SEC public companies
    sec_data = subprocess.run(
        ["cut", "-f3-8,20", "sec/sub.txt"], capture_output=True, text=True
    ).stdout
    df_sec = pandas.read_csv(io.StringIO(sec_data), sep="\t", dtype=str)
    df_sec.columns = ["name", "sic", "country", "state", "city", "zip", "ein"]

    # IRS non-profits
    df_irs = pandas.concat(
        [
            pandas.read_csv(f"irs/eo{i}.csv", usecols=[0, 1, 4, 5, 6], dtype=str)
            for i in range(1, 4)
        ]
    )
    df_irs.columns = df_irs.columns.str.lower()

    # DOL Form 5500
    df_dol = pandas.concat(
        [
            pandas.read_csv(
                "dol_5500/f_5500_2021_latest.csv",
                usecols=[18, 23, 24, 25, 43, 70],
                dtype=str,
                names=["name", "city", "state", "zip", "ein", "size"],
                header=0,
            ),
            pandas.read_csv(
                "dol_5500/f_5500_2020_latest.csv",
                usecols=[18, 23, 24, 25, 43, 70],
                dtype=str,
                names=["name", "city", "state", "zip", "ein", "size"],
                header=0,
            ),
            pandas.read_csv(
                "dol_5500/f_5500_2019_latest.csv",
                usecols=[18, 23, 24, 25, 43, 70],
                dtype=str,
                names=["name", "city", "state", "zip", "ein", "size"],
                header=0,
            ),
            pandas.read_csv(
                "dol_5500/f_5500_sf_2021_latest.csv",
                usecols=[16, 20, 21, 22, 29, 50],
                dtype=str,
                header=0,
                names=["name", "city", "state", "zip", "ein", "size"],
            ),
            pandas.read_csv(
                "dol_5500/f_5500_sf_2020_latest.csv",
                usecols=[16, 20, 21, 22, 29, 50],
                dtype=str,
                header=0,
                names=["name", "city", "state", "zip", "ein", "size"],
            ),
            pandas.read_csv(
                "dol_5500/f_5500_sf_2019_latest.csv",
                usecols=[16, 20, 21, 22, 29, 50],
                dtype=str,
                header=0,
                names=["name", "city", "state", "zip", "ein", "size"],
            ),
        ]
    ).drop_duplicates(subset="ein")
    df_dol["size"] = df_dol["size"].astype("Int64")

    return df_sec, df_irs, df_dol

In [ ]:
#Seperate Dataframes for SEC public companies, IRS Non-Profit and DOL 5500
df_sec, df_irs, df_dol = get_refdata()

## Get the list of all anthem files. Run this code once each month to get the new list of index files.

In [ ]:
# List of all the EINs in anthem
! curl 'https://developers.anthem.com/Resource/GetData?fileType=innetwork&iDisplayStart=1&iDisplayLength=500000' > anthem_file_list.json

In [ ]:
#Read the JSON file list for all CSV GZ files
js_anthem_list = json.load(open("./anthem_file_list.json", "r"))
df_anthem_list = pandas.DataFrame(js_anthem_list["aaData"])
df_anthem_list

## Just curious, what is the total size of anthem data in terabytes.

In [ ]:
# Check total file size
df_anthem_list["size"].sum() / 1e12

## Download anthem index files

In [ ]:
# Only re-run this if you are refreshing the data.
# This could be made faster by running parallel downloads, by submitting batch jobs,
# but we did not want to risk getting our IP banned by the anthem servers, so we downloaded sequentially.
# It will take a several hours to pull these index files.

# Uncomment these lines to re-run download of index files.
# Set the path appropriately.
df_anthem_index_files = df_anthem_list[df_anthem_list.name.str.contains("index")].sort_values("size", ascending=False)
for name in df_anthem_index_files.name.tolist():
    ! curl -s 'https://developers.anthem.com/Resource/DownloadPCTFile?fileType=innetwork&fileName={name}' > /N/u/singrama/Carbonate/Documents/TIC/Richard_Exp/index_json/{name}

## Parse the index files.
## Use multiprocessing to speed up task.

In [ ]:
%%time
#To get details of data table, in_network file & Allowed Amount Table

def batch(iterable, size):
    it = iter(iterable)
    while item := list(itertools.islice(it, size)):
        yield item


def read_index(index_files):
    plan_table = []
    in_network_table = []
    allowed_amount_table = []

    for f in index_files:
        try:
            jo = json.load(open(f, "rb"))
            for rs in jo["reporting_structure"]:
                _plan_table = []
                for rp in rs["reporting_plans"]:
                    rp["fname"] = f.name
                    _plan_table.append(rp)
                plan_table.extend(_plan_table)
                in_network_table.append(
                    {
                        "fname": f.name,
                        "in_network_files": [
                            inf["location"] for inf in rs["in_network_files"]
                        ],
                    }
                )
                allowed_amount_table.append(
                    {
                        "fname": f.name,
                        "allowed_amount_file": rs["allowed_amount_file"]["location"],
                    }
                )
        except:
            # parse fails on a few files that are not well-formed json
            # Note the failures and skip them.
            print(f)
            continue
    return (plan_table, in_network_table, allowed_amount_table)


# Change this to an appropriate location.
anthem_path = pathlib.Path("/N/u/singrama/Carbonate/Documents/TIC/Richard_Exp/index_json")
anthem_index_files = anthem_path.glob("*.json")

# Adjust this to be reasonable. Five cpus is reasonable on RED.
# This will take 10-20 minutes to run.
ncpus = 5
batch_size = 100
with Pool(ncpus) as p:
    result = p.map(read_index, list(batch(list(anthem_index_files), batch_size)))

plan_table = []
in_network_table = []
allowed_amount_table = []

for _pt, _in, _aa in result:
    plan_table.extend(_pt)
    in_network_table.extend(_in)
    allowed_amount_table.extend(_aa)

df_plan_table = pandas.DataFrame(plan_table)
df_in_network_table = pandas.DataFrame(in_network_table)
df_allowed_amount_table = pandas.DataFrame(allowed_amount_table)

## Get Indiana Plans (by EIN of Employer) and Download just those files.

In [ ]:
#Check Plan Table
df_plan_table.head()

In [ ]:
# Function to match EINs from EINs from lists and Insurers
def match_ein(con, df):
    """Search reference data for matching EIN"""
    con.register("df", df)
    return qdf(
        con,
        """
        select distinct 'sec' as tag, df.plan_id, name, city, state from df join df_sec on df.plan_id = df_sec.ein
        union
        select distinct 'irs' as tag, df.plan_id, name, city, state from df join df_irs on df.plan_id = df_irs.ein
        union
        select distinct 'dol' as tag, df.plan_id, name, city, state from df join df_dol on df.plan_id = df_dol.ein
        """,
    )

In [ ]:
# Match the EINs from SEC, IRS and DOLO Form 5500
df_anthem_match = match_ein(con, df_plan_table)

In [ ]:
# Find codes particular to state of Indiana
df_anthem_match.query("state=='IN'")

In [ ]:
# Write out list of Indiana companies to file for reference in txt
df_anthem_match.query("state=='IN'").to_csv(
    "anthem_sample_indiana.txt", index=False, sep="|"
)

In [ ]:
# Just get the plan_ids for Indiana
indiana_plan_ids = df_anthem_match.query("state=='IN'").plan_id.to_list()
len(indiana_plan_ids)

In [ ]:
#Get List of Plan Ids from anthem Data having EIN same as Indiana EINs
indiana_plan_fnames = (
    df_plan_table.query("plan_id.isin(@indiana_plan_ids)").fname.unique().tolist()
)

In [ ]:
#Display final EINs in Indiana
df_plan_table

In [ ]:
# Just get the in-network files for Indiana
indiana_in_network_urls_np = (
    df_in_network_table.query("fname.isin(@indiana_plan_fnames_np)")
    .drop_duplicates("fname")
    .in_network_files.explode()
    .unique()
    .tolist()
)

In [ ]:
ontry = pd.DataFrame(indiana_in_network_urls, columns=['FileName'])
ontry[ontry['FileName']=='https://prod-developers.anthem.com/Resource/DownloadPCTFile?fileType=innetwork&fileName=2022-07-21_2_in-network-rates_000000017710.csv.gz']

In [ ]:
https://prod-developers.anthem.com/Resource/DownloadPCTFile?fileType=innetwork&fileName=2022-07-21_2_in-network-rates_000000017710.csv.gz

In [ ]:
#filename to download from anthem url
def get_fname(url):
    """return filename to download from anthem url"""
    return parse_qs(urlparse(url).query)["fileName"][0]

In [ ]:
# Download one at a time, otherwise the server blocks you.
# The shell script produced below should be run from the command line.
# To speed this up, submit the downloads in this file as separate jobs, 
# but be careful about getting blocked.  Otherwise it just takes a few days.
# run from the terminal:

# $ nohup ./get_anthem_in_network.sh &

scratch_dir = pathlib.Path("/set/your/path")
fout = open("get_anthem_in_network.sh", "w")
for url in indiana_in_network_urls:
    fname = get_fname(url)
    outpath = scratch_dir / fname
    fout.write(f"curl -s '{url}' > {outpath}\n")
fout.close()

In [ ]:
#Get all the CSV files 
path = "/N/project/TIC/anthem/indiana_sample/NPI_Sorted_Indiana_All"
csv_files = glob.glob(os.path.join(path, "*.csv.gz"))

In [ ]:
#Get Key from Value
def get_key(val):
    for key, value in my_dict.items():
        if val == value:
            return key

In [ ]:
#Get Required Billing Codes
bl_dict = {}
#Mention Billing Codes
bl_code = ['99213','45378','73721','27130','80061','99285']
# loop over the list of csv files
for f in tqdm(csv_files): 
    # read the csv file 
    
    df = pd.read_csv(f)
    ounique_bl = list(df.billing_code.unique())

    for bls in ounique_bl:
        if bls in bl_code:
            if bls in bl_dict.keys():
                temp = df[df.billing_code == bls]
                bl_dict[bls] = pd.concat([temp, bl_dict[bls]], axis=0)
            else:
                bl_dict[bls] = df[df.billing_code == bls]

In [ ]:
# Save to CSV 
for v in bl_dict.values():
    billing_code = get_key(v)
    pd.to_csv("Billing_Code_" + billing_code + ".csv" )